# AMRVAC DataDriven Pipeline

Time-dependent B-driven workflow: magnetic sequence -> potential field -> optional magnetofrictional relaxation -> time-dependent magnetofriction or DataDriven MHD. This notebook prepares portable data and AMRVAC cases; run AMRVAC commands in a terminal.

## 1. Setup

Locate AMRVAC and check the Python environment. **Required:** export `AMRVAC_DIR`. **Optional:** use the fallback below only when the environment variable is unavailable. Raw HMI input additionally requires SciPy and SunPy.

In [ ]:
import os
import sys
from pathlib import Path

USER_AMRVAC_DIR = None  # OPTIONAL: Path('/path/to/amrvac') when AMRVAC_DIR is not exported.

amrvac_dir = os.environ.get('AMRVAC_DIR') or USER_AMRVAC_DIR
if amrvac_dir is None:
    raise EnvironmentError('Export AMRVAC_DIR before starting Jupyter, or set USER_AMRVAC_DIR above.')
AMRVAC_ROOT = Path(amrvac_dir).expanduser().resolve()
PYTOOLS = AMRVAC_ROOT / 'tools/python'
if not (PYTOOLS / 'amrvac_pytools').is_dir():
    raise ModuleNotFoundError(f'Cannot locate amrvac_pytools under {PYTOOLS}. Check AMRVAC_DIR.')
if str(PYTOOLS) not in sys.path:
    sys.path.insert(0, str(PYTOOLS))

from amrvac_pytools.datadriven import (
    check_data_driven_dependencies, create_data_driven_workflow, stage_grid_config
)

check_data_driven_dependencies()
print(f'AMRVAC root: {AMRVAC_ROOT}')


## 2. Input And Project

Inspect the first frame and define the portable project. **Required:** magnetic-sequence directory. **Recommended:** project directory and the relaxation/evolution resolution plans. **Optional:** preview limit and allowed block sizes.

In [ ]:
INPUT_DIR = Path('/path/to/magnetic_sequence')  # REQUIRED: raw-HMI or SHARP/CEA sequence directory.
USER_PROJECT_DIR = None  # RECOMMENDED: portable project directory; None uses AMRVAC_ROOT / 'DrivenFieldProject'.
PREVIEW_BMAX = 1000.0  # OPTIONAL: symmetric magnetic-field color limit [G]; None selects it automatically.

RELAXATION_BOUNDARY_REDUCTION_LEVEL = 2  # RECOMMENDED: lower-resolution Potential/MFR boundary.
EVOLUTION_BOUNDARY_REDUCTION_LEVEL = 1  # RECOMMENDED: full-resolution TMF/DataDriven boundary.
EVOLUTION_AMRVAC_REFINEMENT_LEVEL = 2  # RECOMMENDED: finest evolution AMR level.
AMRVAC_BLOCK_SIZES = (12, 14, 16, 18, 20)  # OPTIONAL

workflow = create_data_driven_workflow(
    amrvac_root=AMRVAC_ROOT,
    project_dir=USER_PROJECT_DIR,
    input_dir=INPUT_DIR,
    relaxation_grid=stage_grid_config(
        RELAXATION_BOUNDARY_REDUCTION_LEVEL, 1, AMRVAC_BLOCK_SIZES
    ),
    evolution_grid=stage_grid_config(
        EVOLUTION_BOUNDARY_REDUCTION_LEVEL,
        EVOLUTION_AMRVAC_REFINEMENT_LEVEL,
        AMRVAC_BLOCK_SIZES,
    ),
)
workflow.preview_input(bmax=PREVIEW_BMAX)
print(workflow.input_report())


## 3. Region And Grid Plan

Use one master region for the reference frame and the full sequence. With `REGION_MODE='auto'`, raw HMI uses `RAW_HMI_CEA_PATCH`, while an existing SHARP/CEA map uses `SHARP_PIXEL_WINDOW`; the inactive parameter is ignored. Set `fixed_cea` to force the CEA patch for either input type, or `pixel_window` to force a pixel crop for SHARP/CEA only. **Optional:** automatic block-compatible trimming.

In [ ]:
REGION_MODE = 'auto'  # RECOMMENDED: auto, fixed_cea, or pixel_window.
RAW_HMI_CEA_PATCH = None  # Inactive for the default SHARP test. Example for raw HMI:
# RAW_HMI_CEA_PATCH = {
#     'center_lon': 7.0,
#     'center_lat': 13.0,
#     'width_degree': 40.0,
#     'height_degree': 30.0,
#     'resolution_degree': 0.03,
# }
SHARP_PIXEL_WINDOW = {'x0': 710, 'y0': 0, 'nx': 450, 'ny': 379}  # Active default: right half of the current 379 x 1306 preview.
# Set to None to select the full SHARP/CEA map. Inactive settings are ignored.
AUTO_TRIM_TO_AMRVAC_BLOCKS = True  # OPTIONAL: minimally trim incompatible dimensions.

workflow.plan_region(
    region_mode=REGION_MODE,
    raw_hmi_cea_patch=RAW_HMI_CEA_PATCH,
    sharp_pixel_window=SHARP_PIXEL_WINDOW,
    auto_trim=AUTO_TRIM_TO_AMRVAC_BLOCKS,
)
workflow.plot_region(bmax=PREVIEW_BMAX)
print(workflow.region_report())


## 4. Sequence And Initial Field

Choose the observation interval and the initial/evolution methods. **Recommended:** confirm all five choices. The first selected frame is always the reference frame and is written at observation time `0 s`.

In [ ]:
START_SNAPSHOT_INDEX = 0  # RECOMMENDED: first included observation; indexing starts at 0.
STOP_SNAPSHOT_INDEX = None  # RECOMMENDED: exclusive stop; None includes the remainder of the sequence.
FRAME_STRIDE = 1  # RECOMMENDED: 1 keeps every complete frame.
INITIAL_FIELD = 'potential'  # RECOMMENDED: potential or mfr.
EVOLUTION_MODE = 'tmf'  # RECOMMENDED: tmf or data_driven.

workflow.configure_sequence(
    start_snapshot_index=START_SNAPSHOT_INDEX,
    stop_snapshot_index=STOP_SNAPSHOT_INDEX,
    frame_stride=FRAME_STRIDE,
    initial_field=INITIAL_FIELD,
    evolution_mode=EVOLUTION_MODE,
)
print(workflow.sequence_configuration_report())


## 5. Prepare The Reference Frame And Magnetic Sequence

Stage PotentialField, optionally stage MFR, and write the selected B-only sequence. **Recommended:** confirm the potential solver and resume policy. **Optional:** remap, ghost-cell, preprocessing, MFR, and quicklook settings.

In [ ]:
CEA_REMAP_OPTIONS = {  # OPTIONAL: used for raw HMI -> CEA conversion.
    'sampling_mode': 'sharp',
    'oversample_resolution_degree': 0.01,
    'smooth_sigma_degree': 0.01,
}
BOUNDARY_OPTIONS = {
    'nghost': 2,  # OPTIONAL: ghost cells around the physical magnetogram.
    'preprocess': False,  # OPTIONAL: Cartesian boundary preprocessing.
}
POTENTIAL_OPTIONS = {
    'potential_field_method': 'fft',  # RECOMMENDED: fft or green.
    'fft_padding_factor': 2,  # OPTIONAL
    'lalpha': 0.0,  # OPTIONAL: 0 gives a potential field.
    'fft_top_boundary': 'open',  # OPTIONAL: open or closed.
}
MFR_OPTIONS = {  # OPTIONAL: used only when INITIAL_FIELD == 'mfr'.
    'mf_it_max': 100000,
    'mf_ditsave': 5000,
    'mf_cc': 0.5,
    'mf_cy': 0.2,
    'mf_cdivb': 0.01,
}
RESUME_PREPARED_SEQUENCE = True  # RECOMMENDED: reuse matching raw-HMI conversions.
OVERWRITE_PREPARED_SEQUENCE = True  # OPTIONAL: rebuild only when intentionally changing cached settings.
SHOW_PROGRESS = True  # OPTIONAL: frame-level progress for long raw-HMI sequences.
QUICKLOOK_SNAPSHOT_COUNT = 9  # OPTIONAL: evenly sampled Bz panels; 6 or 9 works well.

workflow.prepare_reference_frame(
    cea_remap_options=CEA_REMAP_OPTIONS,
    potential_options=POTENTIAL_OPTIONS,
    mfr_options=MFR_OPTIONS,
    resume=RESUME_PREPARED_SEQUENCE,
    overwrite=OVERWRITE_PREPARED_SEQUENCE,
    vmax=PREVIEW_BMAX,
    **BOUNDARY_OPTIONS,
)
print(workflow.reference_report())

workflow.prepare_magnetic_sequence(
    cea_remap_options=CEA_REMAP_OPTIONS,
    resume=RESUME_PREPARED_SEQUENCE,
    overwrite=OVERWRITE_PREPARED_SEQUENCE,
    progress=SHOW_PROGRESS,
    vmax=PREVIEW_BMAX,
    **BOUNDARY_OPTIONS,
)
print(workflow.sequence_report())
_sequence_quicklook = workflow.plot_sequence_quicklook(
    vmax=PREVIEW_BMAX,
    snapshot_count=QUICKLOOK_SNAPSHOT_COUNT,
    columns=3,
)


## 6. Run The Initial Field

Run the displayed terminal commands in order. MFR commands appear only when `INITIAL_FIELD='mfr'`. **Recommended:** choose an appropriate MPI process count.

In [ ]:
NPROC = 4  # RECOMMENDED: adjust for the target machine.

print(workflow.initial_field_commands(nproc=NPROC))


## 7. Select The Initial Restart And Stage Evolution

Resolve the potential or MFR restart and stage exactly one evolution case. **Recommended:** confirm the driving time scale and, for DataDriven MHD, the control equations. **Optional:** manually select an MFR restart or atmosphere details.

In [ ]:
SELECTED_RESTART_NUMBER = None  # OPTIONAL for MFR: None uses the relaxation diagnostics.
PLOT_LORENTZ_FORCE = False  # OPTIONAL for MFR: True adds Lorentz force beside sin(theta).
DRIVING_TIME_SCALE = 12.0  # RECOMMENDED: observational time advances 12x faster than simulation time.

TMF_CASE_OPTIONS = {  # OPTIONAL: TMF template overrides.
}
DATA_DRIVEN_CASE_OPTIONS = {
    'mhd_model': 'zero_beta',  # RECOMMENDED: zero_beta, isothermal, adiabatic, or thermodynamic.
    'atmosphere_model': None,  # OPTIONAL: model-dependent default when None.
    'atmosphere_source': 'hydrostatic',  # OPTIONAL: hydrostatic or relaxed_table.
    # USER: density at the physical lower boundary; use None to keep the model normalization.
    'bottom_numberdensity_cm3': 1.0e9,  # Set None when first testing a chromosphere model.
    # 'relaxed_atmosphere_file': Path('/path/to/atmosphere.dat'),  # REQUIRED only for relaxed_table.
    # 'coronal_temperature_k': 1.0e6,  # OPTIONAL
    # 'rho_reference_numberdensity_cm3': 1.0e9,  # OPTIONAL
    # 'temperature_curve': 'AL-C7',  # OPTIONAL for chromosphere.
}

workflow.resolve_initial_restart(
    selected_restart_number=SELECTED_RESTART_NUMBER,
    plot_lorentz_force=PLOT_LORENTZ_FORCE,
)
print(workflow.restart_report())

EVOLUTION_CASE_OPTIONS = (
    TMF_CASE_OPTIONS if EVOLUTION_MODE == 'tmf' else DATA_DRIVEN_CASE_OPTIONS
)
workflow.stage_evolution(
    mode=EVOLUTION_MODE,
    driving_time_scale=DRIVING_TIME_SCALE,
    case_options=EVOLUTION_CASE_OPTIONS,
)
print(workflow.evolution_report())


## 8. Run Evolution

Run the staged TMF or DataDriven case in a terminal. Switching `EVOLUTION_MODE` only requires rerunning Step 7 and this step; the prepared sequence is reused.

In [ ]:
print(workflow.evolution_commands(nproc=NPROC))
